In [ ]:
import pandas as pd
sales = pd.read_csv('sales_data.csv')
#1
(sales
    .groupby('Category')
    .agg(
        T_Qty = pd.NamedAgg(column='Quantity', aggfunc='sum'),
        AVG_Price = pd.NamedAgg(column='Price', aggfunc='mean'),
        Max_Qty = pd.NamedAgg(column='Quantity', aggfunc='max'))
    .reset_index())

In [ ]:
#2
#Groups the dataframe first by categorym=, then by product. After that, returns a series with summed quntities foreach product
sales_grp = sales.groupby(['Category', 'Product']).Quantity.sum()
#Resets indexes category and product to transform back to a dataframe to apply functions like sorting
sales_df = sales_grp.reset_index()
#Sorts values by category and quantity , with biggest quantity numbers at the top
sorted_df = sales_df.sort_values(['Category', 'Quantity'], ascending=[True, False])
#drops duplice rows in category column and keeps the first row, which is in our case the row with highest quantity
sorted_df.drop_duplicates(subset=['Category'], keep='first')

In [ ]:
#3
#Calculates total_sales by multiplying price of a product to its corresponding quantity
sales['Total_Sales'] = sales['Quantity'] * sales['Price']
#Groups by Date and sums all values in total sales in each date group
final = (sales
    .groupby('Date')
    ['Total_Sales']
    .agg('sum')
)
#Sorts resulting values from highest to lowest and selects the first row, which thw row with most sales  
final.sort_values(ascending=False).head(1)

In [ ]:
customer = pd.read_csv('customer_orders.csv')
#4
#Groups the df by customerID and basically counts the number of rows by counting unique identifier OrderID
customer_agg = customer.groupby('CustomerID')['OrderID'].count()
#Checking the resulting series for the condition >= 20
fltr = customer_agg >= 20
#Displaying those customers that satisfy the condition
customer_agg[fltr]

In [ ]:
#5
#Groups df by cutomerID and returns series with customerid as index and price as a mean of price of all products putchased by each customer
customer_agg = customer.groupby('CustomerID')['Price'].mean()
#filter 
fltr = customer_agg > 120
#displaying customers with ans average price per unit greater than 120
customer_agg[fltr]

In [ ]:
#6
#Groups the df by product id and finds the sum of quantity and price
product_agg = customer.groupby('Product')[['Quantity', 'Price']].sum()
#filters out those products that have less than 5 total quantity ordered
fltr = product_agg['Quantity'] >= 5
#applying the filter
product_agg[fltr]

In [ ]:
import pandas as pd
import sqlite3
#7
with sqlite3.connect('population.db') as connection:
    population = pd.read_sql("SELECT * FROM population", connection)

population.set_index(['id'], inplace=True)

In [ ]:
import numpy as np
#8 
# generates a list of labels for grouping 
lab = []
for i in range(1, 11):
    lab.append(f'Group {i}')
#range of values for each group
bins = [0, 200_000, 400_000, 600_000, 800_000, 1_000_000, 1_200_000, 1_400_000, 1_600_000, 1_800_000, float(np.inf)]
#creates a new column 'salary group' with categorized salaries
population['Salary Group'] = pd.cut(population['salary'], bins=bins, labels=lab)
#Groups the df by the newly created column and counts rows int each of the groups and takes one arbitrary column so that it's easier to work with 
population_grp = population.groupby('Salary Group', observed=True).count()['first_name']
#calculates percentages of each group relative to the total population. Basically, find shares of population each group to total number of population
population_pct = (population_grp/population_grp.sum()*100)
#Renames the series so that it makes more sense
population_pct.name = 'Population Percentage'
#removes NaN values so that data is accurate. Then, groups everything into salary groups and calculates the average salary for each group
avg_salary = population.dropna().groupby('Salary Group', observed=True).mean(['salary']).round(2)
#Does the same thing as in the previous task, but finds median rather than mean
median_salary = population.dropna().groupby('Salary Group', observed=True).median(['salary'])
#Groups by salary groups and counts rows taking only one arbitrary column, because all columns show the same data
num_of_category = population.groupby('Salary Group', observed=True).count()['first_name']
#Renames the series to what it actually represents
num_of_category.name = 'Population Number'


In [ ]:
#9
#Everything below is basically the same thing as #8 but grouped by State
state_grp = population.groupby('state')

state_pop_pct = state_grp['first_name'].count()/population['first_name'].count()*100
state_pop_pct.name = 'Population Percent'

state_grp['salary'].mean().round(2)

state_grp['salary'].median()

state_pop = state_grp['first_name'].count()
state_pop.name = 'Number of Population'